# Hello, Model Selection — "the cheapest token is not the cheapest answer"

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/production-ai-engineering/blob/main/notebooks/hello_model_selection.ipynb)

Pairs with **[Lesson 01](../builds/01-model-routing/LESSON.md)**. Three tiers, one decision — and the
surprise that the obvious optimisation (send cheap work to the cheap model) often loses to changing
**one parameter** on a single model.

Self-contained: three synthetic production incidents are embedded, no downloads. You need an
Anthropic API key. **Total spend for the whole notebook: a few cents.**

## 0. Setup

**Two ways to supply your key. Pick either.**

**A — Colab Secrets (recommended).** Click the 🔑 icon in the left sidebar → **Add new secret** →
name it exactly `ANTHROPIC_API_KEY`, paste the value, and turn on **Notebook access**. Do this once
and every future notebook picks it up automatically — nothing to retype.

**B — Type it when prompted.** If no secret is found, the cell below asks for the key and hides it
as you type. It lives only in this session and disappears when the runtime restarts.

Either way the key is **never written into the notebook**, so a saved or shared copy carries no
credential.

Get a key at [console.anthropic.com](https://console.anthropic.com) → **API Keys**. $5 of credit is
far more than this notebook needs — the whole thing costs a few cents.

In [ ]:
!pip install -q anthropic

In [ ]:
import os, time

def load_key():
    """Colab Secrets first, then the environment, then ask.

    Secrets survive a runtime restart; a typed key does not. Reading from the
    environment last-but-one means the same notebook also runs outside Colab.
    """
    try:
        from google.colab import userdata          # only exists inside Colab
        key = userdata.get("ANTHROPIC_API_KEY")
        if key:
            print("key: loaded from Colab Secrets")
            return key
    except Exception:
        pass                                        # not Colab, or secret not enabled

    if os.environ.get("ANTHROPIC_API_KEY"):
        print("key: loaded from the environment")
        return os.environ["ANTHROPIC_API_KEY"]

    import getpass
    print("No secret found — paste your key (it will not be shown or saved).")
    return getpass.getpass("Anthropic API key: ")


os.environ["ANTHROPIC_API_KEY"] = load_key()

import anthropic
client = anthropic.Anthropic()

# Prove the key works before spending anything on the lesson.
print("models reachable:", len(client.models.list().data) > 0)

### The three models, and what they cost

Rates are dollars per million tokens, verified 2026-09-16.

These identifier strings are **complete as written**. Appending a date suffix — a habit from older
model families — produces a model that does not exist.

In [ ]:
OPUS   = "claude-opus-5"
SONNET = "claude-sonnet-5"
HAIKU  = "claude-haiku-4-5"

RATES = {                       # $ per million tokens
    OPUS:   {"in": 5.00, "out": 25.00},
    SONNET: {"in": 2.00, "out": 10.00},
    HAIKU:  {"in": 1.00, "out":  5.00},
}

CONTEXT = {OPUS: 1_000_000, SONNET: 1_000_000, HAIKU: 200_000}


def cost_usd(model, usage):
    """Price a call from what the API says it ACTUALLY used.

    Note this reads usage off the response rather than estimating. The model
    decides how many output tokens to spend, so the only honest cost is reported.
    """
    r = RATES[model]
    return (usage.input_tokens * r["in"] + usage.output_tokens * r["out"]) / 1_000_000


def call(model, system, prompt, max_tokens=600, effort=None):
    """One request, timed and priced."""
    kwargs = dict(model=model, max_tokens=max_tokens, system=system,
                  messages=[{"role": "user", "content": prompt}])
    if effort:                                  # not accepted on every tier
        kwargs["output_config"] = {"effort": effort}

    t0 = time.time()
    resp = client.messages.create(**kwargs)
    secs = time.time() - t0

    text = "".join(b.text for b in resp.content if b.type == "text").strip()
    return {"model": model, "text": text, "secs": secs,
            "in": resp.usage.input_tokens, "out": resp.usage.output_tokens,
            "cost": cost_usd(model, resp.usage)}


def show(r, label=None):
    print(f"  {label or r['model']:<26} {r['secs']:>6.2f}s  "
          f"in={r['in']:>6}  out={r['out']:>5}  ${r['cost']:>9.6f}")

print("helpers defined")

### The data — three production incidents

Synthetic, and from a delivery team's world rather than a textbook. The **difficulty gap** between
them is the whole experiment: if all three were easy, every model would look identical and there
would be nothing to decide.

In [ ]:
SIMPLE = """\
[P4] Disk usage alert on build-agent-07: /var/log at 81% (threshold 80%).
No build failures. Log rotation ran normally at 03:00."""

MODERATE = """\
[P2] Checkout API p99 latency 1.8s -> 6.4s over 40 minutes, error rate 0.3% -> 2.1%.
Deploy of payments-svc v4.12 went out 55 minutes ago. DB connection pool at 94% of max.
Two other services share that pool. No alerts from the payment provider."""

COMPLEX = """\
[P1] Order totals wrong for a subset of EU customers since the 14th.
Finance reports 0.4% of orders under-charged, avg -4.30 EUR; support has 31 tickets.
Three changes landed on the 14th: a tax-rules update, a currency-rounding library bump
(2.1.0 -> 3.0.0), and a caching layer in front of the pricing service.
The rounding library changelog mentions "banker's rounding is now the default".
The cache has a 6-hour TTL. Staging did not reproduce it; staging uses a fixed FX rate.
Rollback of the tax-rules update alone did not fix it."""

for name, text in [("SIMPLE", SIMPLE), ("MODERATE", MODERATE), ("COMPLEX", COMPLEX)]:
    print(f"{name}: {len(text.split())} words")

## 1. One call, and what it cost

The smallest useful thing: send one message, get one answer, find out what you just spent.

**Why this matters:** cost is not a mystery you discover on the invoice. It is reported on every
response and you can read it.

In [ ]:
TRIAGE_ONE_WORD = ("You triage production incidents. "
                   "Reply with exactly one word: LOW, MEDIUM, HIGH, or CRITICAL.")

r = call(HAIKU, TRIAGE_ONE_WORD, SIMPLE, max_tokens=16)

print(f"answer: {r['text']}\n")
show(r, "this call")
print("\nNote how few output tokens a one-word answer needs — and that the cost")
print("came from response.usage, not an estimate.")

## 2. The same work on three tiers

One prompt, one incident, three models. This is what every tier table is really claiming — run for
real.

**Watch two things:**
1. The price gap between tiers is large and obvious.
2. The **answer** gap on a moderate task is often small.

That second observation is what makes routing tempting. Section 3 tests whether it survives.

In [ ]:
TRIAGE_STRUCTURED = (
    "You triage production incidents. Reply with exactly three lines:\n"
    "SEVERITY: <LOW|MEDIUM|HIGH|CRITICAL>\n"
    "LIKELY CAUSE: <one sentence>\n"
    "FIRST ACTION: <one sentence>"
)

results = []
for model in (HAIKU, SONNET, OPUS):
    r = call(model, TRIAGE_STRUCTURED, MODERATE, max_tokens=300)
    results.append(r)
    print(f"--- {model}")
    for line in r["text"].splitlines():
        print(f"    {line}")
    print()

print("Comparison\n")
for r in results:
    show(r)

cheap, dear = min(results, key=lambda x: x["cost"]), max(results, key=lambda x: x["cost"])
print(f"\n{dear['model']} cost {dear['cost'] / cheap['cost']:.1f}x what {cheap['model']} did.")
print("Now read the three answers again. Is it one answer's worth of better?")

## 3. The lever before the second model

Section 2 made routing look obvious. Here is the option that tier table never mentions.

**`effort`** trades thoroughness against token spend *within one model*. Same model, same prompt,
three settings. It costs one line of code and adds **no second model, no classifier, no second
cache**.

If a strong model at low effort lands where you need it, you have your saving — and you still have
one system with one set of failure modes.

> **Measure this before you build a router. That is the entire lesson.**

*Note: `effort` is not accepted on every tier — the small tier rejects it. This uses Opus.*

In [ ]:
HYPOTHESIS = ("You are a staff engineer triaging a production incident. "
              "Give your leading hypothesis and the single next diagnostic step.")

effort_runs = []
for effort in ("low", "medium", "high"):
    r = call(OPUS, HYPOTHESIS, COMPLEX, max_tokens=1200, effort=effort)
    effort_runs.append((effort, r))
    print(f"--- effort={effort}")
    for line in r["text"].splitlines()[:6]:
        print(f"    {line}")
    print()

print("Comparison\n")
for effort, r in effort_runs:
    show(r, f"opus @ {effort}")

low, high = effort_runs[0][1], effort_runs[-1][1]
print(f"\nLow effort cost {low['cost'] / high['cost'] * 100:.0f}% of high effort on the SAME model.")
print("Thinking tokens bill as output, so effort moves the output side.")
print("\nThe question for your workload: did low effort get the RIGHT hypothesis?")

## 4. A router that tells you why

Now build the thing the tier table implies. Two details most routing examples skip, and both are
the point:

1. **The classifier is a model call.** It has a price, charged on **every** request — including
   the ones that route to the expensive model anyway. Never fold it into the routed call.
2. **Every decision reports its rule.** A bare model name is an instruction; a model name plus its
   rule is an *argument*, and an argument can be reviewed by the person whose budget it is.

The policy is **data**. Edit the list, change the behaviour — no other code changes.

In [ ]:
CLASSIFIER = ("Rate how hard this production incident is to diagnose, 1 to 5. "
              "1 = a single obvious cause. 5 = several interacting changes with no clear culprit. "
              "Reply with only the digit.")

# First match wins, so the ORDER is part of the policy.
POLICY = [
    {"id": "R1-trivial",  "max": 2, "model": HAIKU,
     "why": "A single obvious cause does not repay a larger model."},
    {"id": "R2-moderate", "max": 3, "model": SONNET,
     "why": "Mid-difficulty work where the mid tier holds the quality bar."},
    {"id": "R3-default",  "max": 5, "model": OPUS,
     "why": "Interacting causes need the strongest model. The default is capability."},
]


def route(incident):
    c = call(HAIKU, CLASSIFIER, incident, max_tokens=8)
    digits = [ch for ch in c["text"] if ch.isdigit()]
    complexity = int(digits[0]) if digits else 3       # unreadable -> assume middle

    rule = next((r for r in POLICY if complexity <= r["max"]), POLICY[-1])

    # A context window is a HARD constraint, not a preference. A model that cannot
    # hold the input is not a cheaper option -- it is a different failure.
    approx_tokens = len(incident.split()) * 4 // 3
    if approx_tokens > CONTEXT[rule["model"]]:
        rule = {**rule, "id": rule["id"] + "+context-guard", "model": OPUS}

    return complexity, rule, c["cost"]


routed_total = classifier_total = 0.0
for name, incident in [("SIMPLE", SIMPLE), ("MODERATE", MODERATE), ("COMPLEX", COMPLEX)]:
    complexity, rule, classifier_cost = route(incident)
    r = call(rule["model"], HYPOTHESIS, incident, max_tokens=600)
    routed_total += r["cost"]; classifier_total += classifier_cost

    print(f"{name}")
    print(f"  complexity : {complexity}")
    print(f"  rule       : {rule['id']} — {rule['why']}")
    print(f"  model      : {rule['model']}")
    print(f"  routed     : ${r['cost']:.6f}")
    print(f"  classifier : ${classifier_cost:.6f}   <- charged even when routing to opus")
    print()

total = routed_total + classifier_total
print(f"routed calls : ${routed_total:.6f}")
print(f"classifier   : ${classifier_total:.6f}  ({classifier_total / total * 100:.1f}% of the bill)")
print(f"TOTAL        : ${total:.6f}")

## 5. Price it before you send it

Everything so far measured cost **after** spending it. The token-counting endpoint prices the input
side **before** any generation happens — free, and instant.

Why it matters: for long-context work, input cost dominates. A 200,000-token document sent to the
wrong tier is a decision you can evaluate *before* you make it, not one you discover on the invoice.

Only the **output** side needs measuring, because only the model decides that.

In [ ]:
counted = client.messages.count_tokens(
    model=OPUS,
    system=HYPOTHESIS,
    messages=[{"role": "user", "content": COMPLEX}],
)
n = counted.input_tokens
print(f"input tokens (counted, not estimated): {n}\n")

print(f"{'model':<24}{'input cost':>13}{'+1k output':>13}")
print("-" * 50)
for m in (HAIKU, SONNET, OPUS):
    in_cost  = n * RATES[m]["in"] / 1_000_000
    out_cost = 1_000 * RATES[m]["out"] / 1_000_000
    print(f"{m:<24}${in_cost:>12.6f}${in_cost + out_cost:>12.6f}")

print("\nMultiply by your request volume and you have a monthly figure")
print("before writing a line of integration code.")

## 6. What the bench says

Those five sections used real API calls. The **full comparison** — 200 tasks across seven
configurations, including the two costs you cannot see from a single call — is pure arithmetic and
runs with **no API key at all**:

```bash
git clone https://github.com/sunilmogadati/production-ai-engineering.git
cd production-ai-engineering/builds/01-model-routing/src && python3 bench.py
```

| Configuration | Total |
|---|---|
| router (policy cascade) | $20.44 |
| opus only @ high effort | $21.61 |
| **opus only @ low effort** | **$19.89** |
| **sonnet only @ high** | **$12.18** |
| haiku only @ high | $15.16 |
| router, cache reuse off | $21.98 |

**The router beat the naive baseline by 5.4% — and lost to the same model at low effort.** The
cache split it caused cost 7.5%, more than routing saved. And the cheapest tokens (Haiku) produced
an expensive outcome: 48.7% of its bill was escalation.

### What to take away

You will read everywhere that routing saves **80% or more**. That is achievable — when traffic is
almost all trivial, there is no shared prefix to split, and nothing escalates. Break any one and
the number collapses.

**80% is a property of a workload, not a fact about routing.** Measure yours.

### Your turn

Price your team's highest-volume LLM workload using section 5. Bring back three numbers: input
tokens per request, requests per month, and the monthly cost on each tier. No routing, no extra code.